Plik generujący embeddingi MolFormer dla wybranych endpointów w celu wykorzystania ich do trenowania modeli.


In [ ]:
!pip install numpy PyTDC transformers torch tqdm -q

In [ ]:
import torch
from transformers import AutoModel, AutoTokenizer
from tdc.single_pred import ADME
import pandas as pd
import numpy as np
from tqdm import tqdm

#Ładowanie modelu MoLFormer
print("Ładowanie modelu MoLFormer...")
checkpoint = "ibm-research/MoLFormer-XL-both-10pct"

try:
    # Bardzo ważne: trust_remote_code=True pobiera skrypty modelujące z HF
    tokenizer = AutoTokenizer.from_pretrained(checkpoint, trust_remote_code=True)
    model = AutoModel.from_pretrained(checkpoint, trust_remote_code=True)

except Exception as e:
    print(f"Błąd: {e}")

def get_molformer_embedding(smiles_list, batch_size=32):
    all_embeddings = []

    for i in tqdm(range(0, len(smiles_list), batch_size)):
        batch_smiles = smiles_list[i:i+batch_size]

        # Tokenizacja
        inputs = tokenizer(batch_smiles, return_tensors="pt", padding=True, truncation=True).to(device) #zamienia smiles na wektory liczb (pt - python tensors)

        with torch.no_grad():#wyłączenie gradientów bo nie trenujemy modelu
            outputs = model(**inputs) #przepuszcza tokeny przez warstwy sieci MoLFormer

            # MoLFormer zwraca ukryte stany.
            # Najczęstszą metodą jest branie średniej z ostatniej warstwy (mean pooling) - zawsze da nam wektor o stałej długości
            embeddings = outputs.last_hidden_state.mean(dim=1).cpu().numpy()
            all_embeddings.append(embeddings)

    return np.vstack(all_embeddings)



In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Używam urządzenia: {device}")
model = model.to(device)
model.eval()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

# Definiowanie ścieżki do folderu
folder_path = '/content/drive/MyDrive/data_splits'

Edpoint 1:

In [ ]:
# Ładowanie danych z TDC
data = ADME(name='Solubility_AqSolDB')
df = data.get_data()
# df zawiera kolumny 'Drug_ID', 'Drug' (SMILES), 'y' (rozpuszczalność)

In [ ]:
#Generowanie
smiles_list = df['Drug'].tolist()
embeddings = get_molformer_embedding(smiles_list)


In [ ]:
#zapis do pliku
df_embeddings = pd.DataFrame(embeddings, columns=[f'emb_{i}' for i in range(embeddings.shape[1])])
final_df = pd.concat([df[['Drug', 'Y']], df_embeddings], axis=1)

print(f"Wygenerowano embeddingi o rozmiarze: {embeddings.shape}")
file_path = os.path.join(folder_path, 'Solubility_AqSolDB_MoLFormer_embeddings.csv')
final_df.to_csv(file_path, index=False)

print(f"Plik został zapisany w: {file_path}")

Endpoint 2: Caco-2 (Wang)

In [ ]:
# Ładowanie danych z TDC
from tdc.single_pred import ADME
data = ADME(name='Caco2_Wang')
df = data.get_data()

In [ ]:
#Generowanie
smiles_list = df['Drug'].tolist()
embeddings = get_molformer_embedding(smiles_list)


In [ ]:
#zapis do pliku
df_embeddings = pd.DataFrame(embeddings, columns=[f'emb_{i}' for i in range(embeddings.shape[1])])
final_df = pd.concat([df[['Drug', 'Y']], df_embeddings], axis=1)

print(f"Wygenerowano embeddingi o rozmiarze: {embeddings.shape}")
file_path = os.path.join(folder_path, 'Caco2_Wang_MoLFormer_embeddings.csv')
final_df.to_csv(file_path, index=False)

print(f"Plik został zapisany w: {file_path}")

Endpoint 3: Lipophilicity (AstraZeneca)

In [ ]:
# Ładowanie danych z TDC
from tdc.single_pred import ADME
data = ADME(name='Lipophilicity_AstraZeneca')
df = data.get_data()

In [ ]:
#Generowanie
smiles_list = df['Drug'].tolist()
embeddings = get_molformer_embedding(smiles_list)


In [ ]:
#zapis do pliku
df_embeddings = pd.DataFrame(embeddings, columns=[f'emb_{i}' for i in range(embeddings.shape[1])])
final_df = pd.concat([df[['Drug', 'Y']], df_embeddings], axis=1)

print(f"Wygenerowano embeddingi o rozmiarze: {embeddings.shape}")
file_path = os.path.join(folder_path, 'Lipophilicity_AstraZeneca_MoLFormer_embeddings.csv')
final_df.to_csv(file_path, index=False)

print(f"Plik został zapisany w: {file_path}")

Endpoint 4: HIA (Hou)

In [ ]:
# Ładowanie danych z TDC
from tdc.single_pred import ADME
data = ADME(name='HIA_Hou')
df = data.get_data()

In [ ]:
#Generowanie
smiles_list = df['Drug'].tolist()
embeddings = get_molformer_embedding(smiles_list)


In [ ]:
#zapis do pliku
df_embeddings = pd.DataFrame(embeddings, columns=[f'emb_{i}' for i in range(embeddings.shape[1])])
final_df = pd.concat([df[['Drug', 'Y']], df_embeddings], axis=1)

print(f"Wygenerowano embeddingi o rozmiarze: {embeddings.shape}")
file_path = os.path.join(folder_path, 'HIA_Hou_MoLFormer_embeddings.csv')
final_df.to_csv(file_path, index=False)

print(f"Plik został zapisany w: {file_path}")

Endpoint 5: Half Life (Obach)

In [ ]:
# Ładowanie danych z TDC
from tdc.single_pred import ADME
data = ADME(name='Half_Life_Obach')
df = data.get_data()

In [ ]:
#Generowanie
smiles_list = df['Drug'].tolist()
embeddings = get_molformer_embedding(smiles_list)


In [ ]:
#zapis do pliku
df_embeddings = pd.DataFrame(embeddings, columns=[f'emb_{i}' for i in range(embeddings.shape[1])])
final_df = pd.concat([df[['Drug', 'Y']], df_embeddings], axis=1)

print(f"Wygenerowano embeddingi o rozmiarze: {embeddings.shape}")
file_path = os.path.join(folder_path, 'Half_Life_Obach_MoLFormer_embeddings.csv')
final_df.to_csv(file_path, index=False)

print(f"Plik został zapisany w: {file_path}")

Endpoint 6: Clearance Hepatocyte (AZ)

In [ ]:
# Ładowanie danych z TDC
from tdc.single_pred import ADME
data = ADME(name='Clearance_Hepatocyte_AZ')
df = data.get_data()

In [ ]:
#Generowanie
smiles_list = df['Drug'].tolist()
embeddings = get_molformer_embedding(smiles_list)


In [ ]:
#zapis do pliku
df_embeddings = pd.DataFrame(embeddings, columns=[f'emb_{i}' for i in range(embeddings.shape[1])])
final_df = pd.concat([df[['Drug', 'Y']], df_embeddings], axis=1)

print(f"Wygenerowano embeddingi o rozmiarze: {embeddings.shape}")
file_path = os.path.join(folder_path, 'Clearance_Hepatocyte_AZ_MoLFormer_embeddings.csv')
final_df.to_csv(file_path, index=False)

print(f"Plik został zapisany w: {file_path}")

Endpoint 7: CYP3A4 Inhibition (Veith)

In [ ]:
# Ładowanie danych z TDC
from tdc.single_pred import ADME
data = ADME(name='CYP3A4_Veith')
df = data.get_data()

In [ ]:
#Generowanie
smiles_list = df['Drug'].tolist()
embeddings = get_molformer_embedding(smiles_list)


In [ ]:
#zapis do pliku
df_embeddings = pd.DataFrame(embeddings, columns=[f'emb_{i}' for i in range(embeddings.shape[1])])
final_df = pd.concat([df[['Drug', 'Y']], df_embeddings], axis=1)

print(f"Wygenerowano embeddingi o rozmiarze: {embeddings.shape}")
file_path = os.path.join(folder_path, 'CYP3A4_Veith_MoLFormer_embeddings.csv')
final_df.to_csv(file_path, index=False)

print(f"Plik został zapisany w: {file_path}")

Endpoint 8: VDss (Lombardo)

In [ ]:
# Ładowanie danych z TDC
from tdc.single_pred import ADME
data = ADME(name='VDss_Lombardo')
df = data.get_data()

In [ ]:
#Generowanie
smiles_list = df['Drug'].tolist()
embeddings = get_molformer_embedding(smiles_list)


In [ ]:
#zapis do pliku
df_embeddings = pd.DataFrame(embeddings, columns=[f'emb_{i}' for i in range(embeddings.shape[1])])
final_df = pd.concat([df[['Drug', 'Y']], df_embeddings], axis=1)

print(f"Wygenerowano embeddingi o rozmiarze: {embeddings.shape}")
file_path = os.path.join(folder_path, 'VDss_Lombardo_MoLFormer_embeddings.csv')
final_df.to_csv(file_path, index=False)

print(f"Plik został zapisany w: {file_path}")

Endpoint 9: AMES Mutagenicity

In [ ]:
# Ładowanie danych z TDC
from tdc.single_pred import Tox
data = Tox(name='AMES')
df = data.get_data()

In [ ]:
#Generowanie
smiles_list = df['Drug'].tolist()
embeddings = get_molformer_embedding(smiles_list)


In [ ]:
#zapis do pliku
df_embeddings = pd.DataFrame(embeddings, columns=[f'emb_{i}' for i in range(embeddings.shape[1])])
final_df = pd.concat([df[['Drug', 'Y']], df_embeddings], axis=1)

print(f"Wygenerowano embeddingi o rozmiarze: {embeddings.shape}")
file_path = os.path.join(folder_path, 'AMES_MoLFormer_embeddings.csv')
final_df.to_csv(file_path, index=False)

print(f"Plik został zapisany w: {file_path}")

Endpoint 10: hERG (Wang) - NEGATYWNY


In [ ]:
from tdc.single_pred import Tox

# Ładowanie danych z TDC
data = Tox(name='hERG')
df = data.get_data()

# Generowanie embeddingów
smiles_list = df['Drug'].tolist()
embeddings = get_molformer_embedding(smiles_list)

# Konwersja embeddingów do DataFrame
df_embeddings = pd.DataFrame(embeddings, columns=[f'emb_{i}' for i in range(embeddings.shape[1])])

# Zapis do pliku
final_df = pd.concat([df[['Drug', 'Y']], df_embeddings], axis=1)

print(f"Wygenerowano embeddingi o rozmiarze: {embeddings.shape}")
file_path = os.path.join(folder_path, 'hERG_MoLFormer_embeddings.csv')
final_df.to_csv(file_path, index=False)

print(f"Plik został zapisany w: {file_path}")

Endpoint 11: Pgp inhibition

In [ ]:
from tdc.single_pred import ADME

# Ładowanie danych z TDC
data = ADME(name='Pgp_Broccatelli')
df = data.get_data()

# Generowanie embeddingów
smiles_list = df['Drug'].tolist()
embeddings = get_molformer_embedding(smiles_list)

# Konwersja embeddingów do DataFrame
df_embeddings = pd.DataFrame(embeddings, columns=[f'emb_{i}' for i in range(embeddings.shape[1])])

# Zapis do pliku
final_df = pd.concat([df[['Drug', 'Y']], df_embeddings], axis=1)

print(f"Wygenerowano embeddingi o rozmiarze: {embeddings.shape}")
file_path = os.path.join(folder_path, 'Pgp_Broccatelli_MoLFormer_embeddings.csv')
final_df.to_csv(file_path, index=False)

print(f"Plik został zapisany w: {file_path}")

Endppoint 12: CYP2D6 Inhibition

In [ ]:
from tdc.single_pred import ADME

# Ładowanie danych z TDC
data = ADME(name='CYP2D6_Veith')
df = data.get_data()

# Generowanie embeddingów
smiles_list = df['Drug'].tolist()
embeddings = get_molformer_embedding(smiles_list)

# Konwersja embeddingów do DataFrame
df_embeddings = pd.DataFrame(embeddings, columns=[f'emb_{i}' for i in range(embeddings.shape[1])])

# Zapis do pliku
final_df = pd.concat([df[['Drug', 'Y']], df_embeddings], axis=1)

print(f"Wygenerowano embeddingi o rozmiarze: {embeddings.shape}")
file_path = os.path.join(folder_path, 'CYP2D6_Veith_MoLFormer_embeddings.csv')
final_df.to_csv(file_path, index=False)

print(f"Plik został zapisany w: {file_path}")